# Proyecto Integrador de Minería de Datos I  
## Notebook 01 — Inspección inicial del dataset

**Carrera:** Tecnicatura Superior en Ciencia de Datos e Inteligencia Artificial  
**Asignatura:** Minería de Datos I  
**Dataset:** `streaming_users_dirty.json`  
**Integrantes:** Thir Ferreyra Nadia Lorena y Constantinidi Leandro Exequiel  
**Comisión:** Turno Tarde  
**Fecha de análisis:** 27 de junio de 2026  

---

### Propósito de esta etapa

En este notebook se realiza una primera revisión del dataset proporcionado por la cátedra. El objetivo es conocer:

- qué representa cada registro;
- cuántas filas y columnas contiene;
- qué tipo de información almacena cada variable;
- cuántos valores faltantes y duplicados existen;
- qué inconsistencias o valores sospechosos requieren una revisión posterior.

> **Importante:** en esta etapa no se limpia ni se modifica el dataset.  
> Primero se reúne evidencia. Las decisiones definitivas se tomarán y justificarán en `02_calidad_y_limpieza.ipynb`.


## 1. Importación de librerías

Se utiliza:

- `pandas` para cargar, organizar y resumir los datos;
- `numpy` para trabajar con valores numéricos;
- `pathlib` para localizar el archivo de forma reproducible.

No se aplican transformaciones sobre el dataset original.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 2. Localización y carga del dataset original

El notebook está pensado para ejecutarse desde la carpeta `notebooks/` del repositorio.  
Se buscan varias rutas posibles para que también pueda abrirse desde la raíz del proyecto.

El archivo se carga directamente desde `data/raw/`, porque esa carpeta debe conservar el dataset original sin modificaciones.


In [2]:
rutas_posibles = [
    Path("../data/raw/streaming_users_dirty.json"),
    Path("data/raw/streaming_users_dirty.json"),
    Path("/content/PI_Mineria_Datos_1/data/raw/streaming_users_dirty.json"),
]

ruta_datos = None

for ruta in rutas_posibles:
    if ruta.exists():
        ruta_datos = ruta
        break

if ruta_datos is None:
    raise FileNotFoundError(
        "No se encontró streaming_users_dirty.json. "
        "Verifique que esté guardado en data/raw/."
    )

df = pd.read_json(ruta_datos)

# Copia de control para comprobar al final que no se modificó el original.
df_original = df.copy(deep=True)

print(f"Archivo cargado desde: {ruta_datos.resolve()}")
print("Carga finalizada correctamente.")

Archivo cargado desde: /mnt/data/PI_Mineria_Datos_1_GRUPAL/data/raw/streaming_users_dirty.json
Carga finalizada correctamente.


### ¿Por qué se crea `df_original`?

Se crea una copia únicamente como control. Al terminar el notebook se comparará con `df` para comprobar que la inspección no alteró los datos.

Esto ayuda a mantener la trazabilidad: el primer notebook debe observar el dataset, no corregirlo.


## 3. Primera visualización de los registros

Se muestran las primeras y las últimas filas para observar:

- los nombres de las columnas;
- el aspecto general de los valores;
- si los registros mantienen una estructura uniforme;
- si al final del archivo aparecen situaciones diferentes de las del comienzo.


In [3]:
print("Primeras cinco filas:")
display(df.head())

print("\nÚltimas cinco filas:")
display(df.tail())

Primeras cinco filas:


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.80,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,"1,173.40",Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.00,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.40,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.80,Perú,Thriller,2020-09-30,1



Últimas cinco filas:


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
8155,10923,23,Premium,"1,161.40",Colombia,Romance,2023-05-15,0
8156,16525,27,Básico,436.20,Uruguay,Documental,2018-09-06,4
8157,11222,13,Estándar,"1,321.80",México,Documental,2019-02-08,0
8158,15613,38,Estándar,835.70,Brasil,Drama,2022-02-05,0
8159,16912,25,Estándar,"1,468.70",Argentina,Romance,2022-08-12,3


## 4. Dimensiones y estructura general

`shape` devuelve dos valores:

1. cantidad de filas;
2. cantidad de columnas.

Cada fila debería representar a un usuario de la plataforma y cada columna una característica de ese usuario.


In [4]:
cantidad_filas, cantidad_columnas = df.shape

print(f"Cantidad de filas: {cantidad_filas:,}")
print(f"Cantidad de columnas: {cantidad_columnas}")
print(f"Cantidad de usuarios identificados: {df['user_id'].nunique():,}")

Cantidad de filas: 8,160
Cantidad de columnas: 8
Cantidad de usuarios identificados: 8,000


### Interpretación inicial

El número de filas no coincide con la cantidad de identificadores únicos. Esto indica que algunos `user_id` aparecen más de una vez.

Todavía no se elimina ningún registro porque primero se debe examinar si las repeticiones son copias exactas o si contienen diferencias.


## 5. Diccionario inicial de variables

La siguiente tabla describe el significado analítico de cada columna. También diferencia entre el tipo detectado por Python y el tipo conceptual esperado.


In [5]:
diccionario_variables = pd.DataFrame({
    "variable": [
        "user_id",
        "age",
        "subscription_plan",
        "monthly_watch_time_mins",
        "country",
        "favorite_genre",
        "last_login_date",
        "customer_support_tickets",
    ],
    "significado": [
        "Identificador único del usuario",
        "Edad declarada del usuario",
        "Plan de suscripción contratado",
        "Minutos de visualización acumulados en un mes",
        "País informado para el usuario",
        "Género de contenido preferido",
        "Fecha del último ingreso a la plataforma",
        "Cantidad de tickets enviados al soporte",
    ],
    "tipo_conceptual_esperado": [
        "Identificador",
        "Numérica discreta",
        "Categórica ordinal",
        "Numérica continua",
        "Categórica nominal",
        "Categórica nominal",
        "Fecha",
        "Numérica discreta",
    ],
    "tipo_detectado_por_pandas": [str(df[col].dtype) for col in df.columns],
})

display(diccionario_variables)

,variable,significado,tipo_conceptual_esperado,tipo_detectado_por_pandas
0,user_id,Identificador único del usuario,Identificador,int64
1,age,Edad declarada del usuario,Numérica discreta,int64
2,subscription_plan,Plan de suscripción contratado,Categórica ordinal,object
3,monthly_watch_time_mins,Minutos de visualización acumulados en un mes,Numérica continua,float64
4,country,País informado para el usuario,Categórica nominal,object
5,favorite_genre,Género de contenido preferido,Categórica nominal,object
6,last_login_date,Fecha del último ingreso a la plataforma,Fecha,object
7,customer_support_tickets,Cantidad de tickets enviados al soporte,Numérica discreta,int64


### Observación sobre los tipos

`last_login_date` fue detectada como `object`, no como fecha. Esto no significa que todos sus valores sean incorrectos: significa que todavía están almacenados como texto y que algunos formatos podrían ser inconsistentes.

La conversión definitiva se realizará en el notebook de calidad y limpieza. Aquí solo se medirá el problema.


## 6. Información técnica del DataFrame

`info()` permite revisar simultáneamente:

- nombres de las columnas;
- cantidad de valores no nulos;
- tipos de datos;
- memoria utilizada.


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   object 
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   object 
 5   favorite_genre            7920 non-null   object 
 6   last_login_date           7840 non-null   object 
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 510.1+ KB


## 7. Valores faltantes

Se calcula la cantidad y el porcentaje de valores faltantes por columna.

El porcentaje permite evaluar la magnitud del problema sin depender únicamente de la cantidad absoluta.


In [7]:
faltantes = pd.DataFrame({
    "cantidad_faltante": df.isna().sum(),
    "porcentaje_faltante": (df.isna().mean() * 100).round(2),
}).sort_values("cantidad_faltante", ascending=False)

display(faltantes)

,cantidad_faltante,porcentaje_faltante
last_login_date,320,3.92
favorite_genre,240,2.94
monthly_watch_time_mins,193,2.37
user_id,0,0.00
subscription_plan,0,0.00
age,0,0.00
country,0,0.00
customer_support_tickets,0,0.00


### Interpretación de los faltantes

Los valores faltantes se concentran en:

- `last_login_date`;
- `favorite_genre`;
- `monthly_watch_time_mins`.

En esta etapa no se imputan ni eliminan. La cantidad por sí sola no indica cuál es el tratamiento correcto. En el próximo notebook se deberá analizar el significado de cada ausencia y el riesgo de introducir sesgo.


## 8. Registros duplicados

Se revisan dos situaciones diferentes:

1. **filas completamente idénticas**;
2. **identificadores repetidos**, aunque alguna otra columna sea diferente.

Esta distinción es importante porque una copia exacta y dos versiones distintas de un mismo usuario no representan el mismo problema.


In [8]:
duplicados_exactos = df.duplicated().sum()
identificadores_repetidos_extra = df.duplicated(subset="user_id").sum()
filas_con_id_repetido = df.duplicated(subset="user_id", keep=False).sum()
usuarios_con_id_repetido = df.loc[
    df.duplicated(subset="user_id", keep=False), "user_id"
].nunique()

print(f"Filas completamente duplicadas: {duplicados_exactos:,}")
print(f"Repeticiones adicionales de user_id: {identificadores_repetidos_extra:,}")
print(f"Filas involucradas en user_id repetidos: {filas_con_id_repetido:,}")
print(f"Usuarios con user_id repetido: {usuarios_con_id_repetido:,}")

Filas completamente duplicadas: 126
Repeticiones adicionales de user_id: 160
Filas involucradas en user_id repetidos: 320
Usuarios con user_id repetido: 160


### Comparación de las dos versiones de cada usuario repetido

El siguiente bloque no corrige los duplicados. Solo compara las dos apariciones y registra en qué columnas difieren.


In [9]:
df_ids_repetidos = (
    df[df.duplicated(subset="user_id", keep=False)]
    .sort_values(["user_id"])
)

comparaciones = []

for user_id, grupo in df_ids_repetidos.groupby("user_id"):
    fila_1 = grupo.iloc[0]
    fila_2 = grupo.iloc[1]
    columnas_diferentes = []

    for columna in df.columns:
        valor_1 = fila_1[columna]
        valor_2 = fila_2[columna]

        ambos_nulos = pd.isna(valor_1) and pd.isna(valor_2)
        mismo_valor = valor_1 == valor_2 if not ambos_nulos else True

        if not mismo_valor:
            columnas_diferentes.append(columna)

    comparaciones.append({
        "user_id": user_id,
        "cantidad_columnas_diferentes": len(columnas_diferentes),
        "columnas_diferentes": ", ".join(columnas_diferentes)
        if columnas_diferentes else "Ninguna: copia exacta",
    })

comparacion_duplicados = pd.DataFrame(comparaciones)

print("Resumen de usuarios repetidos según cantidad de diferencias:")
display(
    comparacion_duplicados["cantidad_columnas_diferentes"]
    .value_counts()
    .sort_index()
    .rename_axis("cantidad_columnas_diferentes")
    .reset_index(name="cantidad_usuarios")
)

print("\nColumnas en las que aparecen diferencias:")
columnas_con_diferencias = (
    comparacion_duplicados.loc[
        comparacion_duplicados["cantidad_columnas_diferentes"] > 0,
        "columnas_diferentes",
    ]
    .value_counts()
    .reset_index()
)
columnas_con_diferencias.columns = ["columnas_diferentes", "cantidad_usuarios"]
display(columnas_con_diferencias)

print("\nEjemplos de usuarios duplicados con diferencias:")
display(
    comparacion_duplicados[
        comparacion_duplicados["cantidad_columnas_diferentes"] > 0
    ].head(10)
)

Resumen de usuarios repetidos según cantidad de diferencias:


,cantidad_columnas_diferentes,cantidad_usuarios
0,0,126
1,1,32
2,2,2



Columnas en las que aparecen diferencias:


,columnas_diferentes,cantidad_usuarios
0,last_login_date,9
1,favorite_genre,6
2,monthly_watch_time_mins,6
3,country,6
4,age,2
5,subscription_plan,2
6,"monthly_watch_time_mins, favorite_genre",1
7,"favorite_genre, last_login_date",1
8,customer_support_tickets,1



Ejemplos de usuarios duplicados con diferencias:


,user_id,cantidad_columnas_diferentes,columnas_diferentes
2,10059,2,"monthly_watch_time_mins, favorite_genre"
16,10721,1,last_login_date
19,10797,1,monthly_watch_time_mins
28,11092,1,favorite_genre
32,11222,1,country
33,11292,1,country
43,11586,1,favorite_genre
47,11746,1,favorite_genre
48,11796,1,last_login_date
60,12634,1,age


### Interpretación de los duplicados

La inspección muestra dos casos:

- una parte de los usuarios repetidos tiene dos filas exactamente iguales;
- otra parte presenta diferencias en una o más columnas.

Por lo tanto, todavía no se debe aplicar `drop_duplicates()` sin especificar un criterio. En el notebook de limpieza habrá que justificar cuál versión conservar, apoyándose en la estructura y el orden del archivo.


## 9. Resumen estadístico de variables numéricas

Se muestran medidas de tendencia central, dispersión y posición.

La comparación entre media, mediana, mínimos, máximos y percentiles ayuda a detectar distribuciones asimétricas y valores que requieren revisión.


In [10]:
columnas_numericas = [
    "age",
    "monthly_watch_time_mins",
    "customer_support_tickets",
]

resumen_numerico = df[columnas_numericas].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

display(resumen_numerico)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
age,"8,160.00",34.10,14.51,-5.00,13.00,13.00,25.00,33.00,42.00,54.00,66.00,150.00
monthly_watch_time_mins,"7,967.00","1,107.35","5,310.44",-120.00,0.00,128.96,489.20,757.40,"1,045.70","1,415.10","3,634.79","99,999.00"
customer_support_tickets,"8,160.00",1.80,11.33,-1.00,0.00,0.00,0.00,1.00,1.00,3.00,4.00,150.00


### Primera lectura del resumen numérico

Se observan máximos y mínimos muy alejados de la mayor parte de los datos:

- edades negativas, iguales a cero o extremadamente altas;
- minutos mensuales negativos y valores muy superiores al comportamiento habitual;
- tickets negativos y cantidades excepcionalmente altas.

Todavía no se afirma que todo valor extremo sea un error. Primero se separarán los valores físicamente imposibles de los valores simplemente poco frecuentes.


## 10. Revisión preliminar de la edad

Para reunir evidencia se muestran los valores extremos y se cuentan los registros fuera del rango preliminar de 13 a 100 años.

La inspección revela valores aislados de -5, 0, 4, 130 y 150 años. No aparecen edades entre 5 y 12, mientras que desde los 13 años comienza una distribución continua. En esta etapa el rango se utiliza solo como regla de revisión; la decisión definitiva se documenta en el notebook de limpieza.

In [11]:
frecuencia_edades = (
    df["age"]
    .value_counts()
    .sort_index()
    .rename_axis("edad")
    .reset_index(name="cantidad")
)

print("Edades más bajas registradas:")
display(frecuencia_edades.head(10))

print("\nEdades más altas registradas:")
display(frecuencia_edades.tail(10))

edades_para_revisar = df[(df["age"] < 13) | (df["age"] > 100)]

print(
    f"Registros fuera del rango preliminar 13-100: "
    f"{len(edades_para_revisar):,} "
    f"({len(edades_para_revisar) / len(df) * 100:.2f}%)"
)

display(
    edades_para_revisar["age"]
    .value_counts()
    .sort_index()
    .rename_axis("edad")
    .reset_index(name="cantidad")
)

Edades más bajas registradas:


,edad,cantidad
0,-5,21
1,0,24
2,4,22
3,13,391
4,14,71
5,15,84
6,16,85
7,17,96
8,18,107
9,19,116



Edades más altas registradas:


,edad,cantidad
59,69,2
60,70,1
61,71,2
62,72,4
63,73,2
64,74,1
65,76,1
66,80,3
67,130,34
68,150,19


Registros fuera del rango preliminar 13-100: 120 (1.47%)


,edad,cantidad
0,-5,21
1,0,24
2,4,22
3,130,34
4,150,19


## 11. Revisión preliminar del tiempo mensual de visualización

Se examinan tres situaciones:

- valores faltantes;
- valores negativos;
- valores superiores a 44.640 minutos, equivalentes a todos los minutos de un mes de 31 días.

El límite mensual se usa como referencia física para detectar registros que requieren investigación. No se modifica ningún valor.


In [12]:
minutos_mes_31_dias = 31 * 24 * 60

cantidad_negativos = (df["monthly_watch_time_mins"] < 0).sum()
cantidad_ceros = (df["monthly_watch_time_mins"] == 0).sum()
cantidad_mayores_mes = (
    df["monthly_watch_time_mins"] > minutos_mes_31_dias
).sum()

print(f"Valores faltantes: {df['monthly_watch_time_mins'].isna().sum():,}")
print(f"Valores negativos: {cantidad_negativos:,}")
print(f"Valores iguales a cero: {cantidad_ceros:,}")
print(
    f"Valores superiores a {minutos_mes_31_dias:,} minutos: "
    f"{cantidad_mayores_mes:,}"
)

print("\nFrecuencia de valores negativos:")
display(
    df.loc[df["monthly_watch_time_mins"] < 0, "monthly_watch_time_mins"]
    .value_counts()
    .sort_index()
    .rename_axis("minutos")
    .reset_index(name="cantidad")
)

print("\nFrecuencia de valores superiores a los minutos de un mes:")
display(
    df.loc[
        df["monthly_watch_time_mins"] > minutos_mes_31_dias,
        "monthly_watch_time_mins",
    ]
    .value_counts()
    .sort_index()
    .rename_axis("minutos")
    .reset_index(name="cantidad")
)

Valores faltantes: 193
Valores negativos: 49
Valores iguales a cero: 167
Valores superiores a 44,640 minutos: 31

Frecuencia de valores negativos:


,minutos,cantidad
0,-120.00,29
1,-1.00,20



Frecuencia de valores superiores a los minutos de un mes:


,minutos,cantidad
0,"50,000.00",11
1,"99,999.00",20


### Detección estadística mediante IQR

El rango intercuartílico permite identificar valores alejados de la zona central de la distribución.

> Un valor marcado por IQR no es automáticamente un error.  
> Puede representar un usuario real con consumo intensivo. La regla sirve para seleccionar casos que merecen análisis.


In [13]:
q1 = df["monthly_watch_time_mins"].quantile(0.25)
q3 = df["monthly_watch_time_mins"].quantile(0.75)
iqr = q3 - q1

limite_inferior_iqr = q1 - 1.5 * iqr
limite_superior_iqr = q3 + 1.5 * iqr

cantidad_outliers_iqr = (
    (df["monthly_watch_time_mins"] < limite_inferior_iqr)
    | (df["monthly_watch_time_mins"] > limite_superior_iqr)
).sum()

print(f"Q1: {q1:,.2f}")
print(f"Q3: {q3:,.2f}")
print(f"IQR: {iqr:,.2f}")
print(f"Límite inferior: {limite_inferior_iqr:,.2f}")
print(f"Límite superior: {limite_superior_iqr:,.2f}")
print(f"Valores señalados por IQR: {cantidad_outliers_iqr:,}")

Q1: 489.20
Q3: 1,045.70
IQR: 556.50
Límite inferior: -345.55
Límite superior: 1,880.45
Valores señalados por IQR: 155


## 12. Revisión preliminar de tickets de soporte

Una variable discreta debería presentar valores enteros no negativos. La tabla de frecuencias permite observar tanto el comportamiento habitual como los saltos inesperados.


In [14]:
frecuencia_tickets = (
    df["customer_support_tickets"]
    .value_counts()
    .sort_index()
    .rename_axis("tickets")
    .reset_index(name="cantidad")
)

display(frecuencia_tickets)

,tickets,cantidad
0,-1,29
1,0,3628
2,1,2885
3,2,1179
4,3,285
5,4,73
6,5,14
7,99,35
8,150,32


### Interpretación inicial

La mayor parte de los usuarios registra entre 0 y 3 tickets. También aparecen valores negativos y un salto marcado hacia 99 y 150.

Esa discontinuidad sugiere que estos casos deben revisarse. En esta etapa no se reemplazan porque todavía debe definirse si son errores, códigos especiales o valores reales.


## 13. Revisión de variables categóricas

Se examinan las frecuencias exactas, respetando mayúsculas, tildes, idioma y espacios.

Esto permite detectar categorías que probablemente representan el mismo concepto, pero fueron escritas de distintas maneras.


In [15]:
columnas_categoricas = [
    "subscription_plan",
    "country",
    "favorite_genre",
]

for columna in columnas_categoricas:
    print(f"\n{'=' * 70}")
    print(f"Variable: {columna}")
    print(f"Categorías distintas, incluyendo nulos: {df[columna].nunique(dropna=False)}")

    frecuencias = (
        df[columna]
        .value_counts(dropna=False)
        .rename_axis(columna)
        .reset_index(name="cantidad")
    )

    # repr() permite hacer visibles espacios al comienzo o al final.
    frecuencias[f"{columna}_visible"] = frecuencias[columna].map(repr)

    display(
        frecuencias[[f"{columna}_visible", "cantidad"]]
    )


Variable: subscription_plan
Categorías distintas, incluyendo nulos: 15


,subscription_plan_visible,cantidad
0,'Básico',3450
1,'Estándar',2711
2,'Premium',1519
3,'basico',60
4,'BASICO',52
5,'Basic',52
6,'básico',50
7,'Std',48
8,'Estándar ',46
9,'estandar',36



Variable: country
Categorías distintas, incluyendo nulos: 26


,country_visible,cantidad
0,'Brasil',1132
1,'Chile',1132
2,'México',1129
3,'Uruguay',1124
4,'Perú',1120
5,'Colombia',1116
6,'Argentina',1087
7,'colombia',27
8,'méxico',25
9,'uruguay',24



Variable: favorite_genre
Categorías distintas, incluyendo nulos: 29


,favorite_genre_visible,cantidad
0,'Comedia',1112
1,'Drama',1105
2,'Documental',1095
3,'Romance',1090
4,'Thriller',1090
5,'Acción',1082
6,'Crime',1067
7,None,240
8,'Action',22
9,'COMEDIA',19


### Interpretación de las categorías

Se observan posibles equivalencias, por ejemplo:

- `Básico`, `basico`, `BASICO` y `Basic`;
- `Brasil`, `brasil`, `Brazil` y `BRA`;
- `Acción`, `ACCIÓN`, `accion` y `Action`;
- valores con espacios al final;
- errores tipográficos como `Premiun` o `thriler`.

Todavía no se unifican. En el próximo notebook se construirá un diccionario explícito para que cada reemplazo quede documentado y sea reproducible.


## 14. Revisión de fechas

La variable se analiza sin reemplazar la columna original.

Se crea una serie temporal auxiliar para medir:

- cuántos valores pueden convertirse a fecha;
- cuántos textos no pueden interpretarse;
- cuántas fechas son posteriores a la fecha de corte del proyecto.

Se utiliza una fecha de corte fija (`2026-06-27`) para que los resultados sean reproducibles.


In [16]:
fecha_corte = pd.Timestamp("2026-06-27")

# Serie auxiliar: no se asigna de vuelta al DataFrame.
fechas_convertidas = pd.to_datetime(
    df["last_login_date"],
    errors="coerce",
    format="mixed",
)

faltantes_originales_fecha = df["last_login_date"].isna().sum()
fechas_no_convertibles_totales = fechas_convertidas.isna().sum()
textos_no_convertibles = (
    fechas_no_convertibles_totales - faltantes_originales_fecha
)
fechas_futuras = (fechas_convertidas > fecha_corte).sum()

print(f"Faltantes originales: {faltantes_originales_fecha:,}")
print(f"Textos no convertibles a fecha: {textos_no_convertibles:,}")
print(f"Fechas posteriores al {fecha_corte.date()}: {fechas_futuras:,}")
print(f"Fecha válida más antigua: {fechas_convertidas.min().date()}")
print(f"Fecha convertida más reciente: {fechas_convertidas.max().date()}")

Faltantes originales: 320
Textos no convertibles a fecha: 64
Fechas posteriores al 2026-06-27: 15


Fecha válida más antigua: 2018-01-01
Fecha convertida más reciente: 2029-01-01


In [17]:
def identificar_patron_fecha(valor):
    if pd.isna(valor):
        return "Nulo"

    texto = str(valor)

    if pd.Series([texto]).str.fullmatch(r"\d{4}-\d{2}-\d{2}").iloc[0]:
        return "AAAA-MM-DD"

    if pd.Series([texto]).str.fullmatch(r"\d{4}/\d{2}/\d{2}").iloc[0]:
        return "AAAA/MM/DD"

    if pd.Series([texto]).str.fullmatch(r"\d{2}-\d{2}-\d{4}").iloc[0]:
        return "DD-MM-AAAA o MM-DD-AAAA"

    return "Otro texto"

patrones_fecha = (
    df["last_login_date"]
    .map(identificar_patron_fecha)
    .value_counts()
    .rename_axis("patron")
    .reset_index(name="cantidad")
)

display(patrones_fecha)

print("\nValores no nulos que no pudieron convertirse:")
valores_no_convertibles = (
    df.loc[
        df["last_login_date"].notna() & fechas_convertidas.isna(),
        "last_login_date",
    ]
    .value_counts()
    .rename_axis("valor_original")
    .reset_index(name="cantidad")
)

display(valores_no_convertibles)

,patron,cantidad
0,AAAA-MM-DD,7428
1,Nulo,320
2,DD-MM-AAAA o MM-DD-AAAA,265
3,AAAA/MM/DD,133
4,Otro texto,14



Valores no nulos que no pudieron convertirse:


,valor_original,cantidad
0,2026-15-03,20
1,0000-00-00,17
2,not_available,14
3,31-02-2022,13


### Interpretación inicial de las fechas

La columna combina diferentes formatos y también contiene:

- valores faltantes;
- textos como `not_available`;
- fechas imposibles;
- fechas futuras.

La conversión con `errors="coerce"` se utilizó solo para medir el problema. En el notebook de limpieza será necesario documentar qué formatos se reconocen y qué hacer con los casos que no pueden reconstruirse.


## 15. Resumen consolidado de problemas detectados

La siguiente tabla reúne evidencia, pero todavía no prescribe una solución definitiva.


In [18]:
resumen_problemas = pd.DataFrame({
    "aspecto_revisado": [
        "Identificadores repetidos",
        "Filas completamente duplicadas",
        "Minutos mensuales faltantes",
        "Género favorito faltante",
        "Fecha de último ingreso faltante",
        "Edades fuera del rango preliminar 13-100",
        "Minutos mensuales negativos",
        "Minutos superiores a un mes de 31 días",
        "Tickets negativos",
        "Tickets iguales a 99 o 150",
        "Textos de fecha no convertibles",
        "Fechas posteriores a la fecha de corte",
        "Planes escritos de formas diferentes",
        "Países escritos de formas diferentes",
        "Géneros escritos de formas diferentes",
    ],
    "evidencia_cuantitativa": [
        identificadores_repetidos_extra,
        duplicados_exactos,
        int(df["monthly_watch_time_mins"].isna().sum()),
        int(df["favorite_genre"].isna().sum()),
        int(df["last_login_date"].isna().sum()),
        int(((df["age"] < 13) | (df["age"] > 100)).sum()),
        int((df["monthly_watch_time_mins"] < 0).sum()),
        int((df["monthly_watch_time_mins"] > minutos_mes_31_dias).sum()),
        int((df["customer_support_tickets"] < 0).sum()),
        int(df["customer_support_tickets"].isin([99, 150]).sum()),
        int(textos_no_convertibles),
        int(fechas_futuras),
        int(df["subscription_plan"].nunique(dropna=False)),
        int(df["country"].nunique(dropna=False)),
        int(df["favorite_genre"].nunique(dropna=False)),
    ],
    "estado_en_este_notebook": [
        "Requiere comparar versiones",
        "Detectado, no eliminado",
        "Detectado, no imputado",
        "Detectado, no imputado",
        "Detectado, no imputado",
        "Marcado para revisión",
        "Marcado para revisión",
        "Marcado para revisión",
        "Marcado para revisión",
        "Marcado para revisión",
        "Detectado, no corregido",
        "Detectado, no corregido",
        "Requiere estandarización",
        "Requiere estandarización",
        "Requiere estandarización",
    ],
})

display(resumen_problemas)

,aspecto_revisado,evidencia_cuantitativa,estado_en_este_notebook
0,Identificadores repetidos,160,Requiere comparar versiones
1,Filas completamente duplicadas,126,"Detectado, no eliminado"
2,Minutos mensuales faltantes,193,"Detectado, no imputado"
3,Género favorito faltante,240,"Detectado, no imputado"
4,Fecha de último ingreso faltante,320,"Detectado, no imputado"
5,Edades fuera del rango preliminar 13-100,120,Marcado para revisión
6,Minutos mensuales negativos,49,Marcado para revisión
7,Minutos superiores a un mes de 31 días,31,Marcado para revisión
8,Tickets negativos,29,Marcado para revisión
9,Tickets iguales a 99 o 150,67,Marcado para revisión


## 16. Preguntas iniciales del proyecto

A partir de la estructura del dataset, se proponen las siguientes preguntas:

1. ¿Cómo se distribuyen los usuarios entre los distintos planes de suscripción?
2. ¿El tiempo mensual de visualización cambia según el plan contratado?
3. ¿Existe una relación entre la edad y el tiempo mensual de visualización?
4. ¿Los patrones de consumo varían entre países una vez considerado el plan?
5. ¿Las variables numéricas contienen información redundante que pueda resumirse mediante PCA?

Estas preguntas son preliminares. Se confirmarán después de verificar que las variables necesarias tengan una calidad suficiente.


## 17. Conclusión de la inspección inicial

El dataset contiene información suficiente para estudiar el comportamiento de usuarios de una plataforma de streaming, pero no está listo para el análisis final.

La evidencia reunida muestra:

- registros duplicados, algunos idénticos y otros con diferencias;
- valores faltantes en tres variables;
- categorías equivalentes escritas con distintas convenciones;
- edades, minutos y tickets que requieren validación;
- fechas con formatos múltiples, valores imposibles y fechas futuras.

La decisión metodológica de este notebook fue **no corregir todavía**. Esto evita aplicar tratamientos automáticos sin comprender primero el problema.

El próximo paso será desarrollar `02_calidad_y_limpieza.ipynb`, donde cada transformación se documentará mediante:

> **Evidencia observada → decisión aplicada → impacto en el dataset.**


## 18. Verificación de integridad

Se comprueba que el DataFrame al final del notebook sea exactamente igual a la copia creada inmediatamente después de la carga.


In [19]:
dataset_sin_modificaciones = df.equals(df_original)

print(f"¿El dataset se mantuvo sin modificaciones? {dataset_sin_modificaciones}")

if not dataset_sin_modificaciones:
    raise AssertionError(
        "El DataFrame fue modificado durante la inspección inicial."
    )

¿El dataset se mantuvo sin modificaciones? True


### Resultado final

La verificación devuelve `True`, por lo que la inspección reunió evidencia sin alterar el dataset original.
